In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load in 

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the "../input/" directory.
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Any results you write to the current directory are saved as output.

/kaggle/input/tagsdesc/tagsdesc.txt
/kaggle/input/arxivdataset/arxivData.json


In [3]:
import re
import spacy
from spacy.lang.en.stop_words import STOP_WORDS
import string
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models import Word2Vec
from gensim.models.phrases import Phraser,Phrases
from time import time
import multiprocessing
from gensim.models.doc2vec import Doc2Vec,TaggedDocument
from IPython.display import HTML

In [4]:
df = pd.read_json('../input/arxivdataset/arxivData.json',orient='records')
df.head()

,author,day,id,link,month,summary,tag,title,year
0,"[{'name': 'Ahmed Osman'}, {'name': 'Wojciech S...",1,1802.00209v1,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",2,We propose an architecture for VQA which utili...,"[{'term': 'cs.AI', 'scheme': 'http://arxiv.org...",Dual Recurrent Attention Units for Visual Ques...,2018
1,"[{'name': 'Ji Young Lee'}, {'name': 'Franck De...",12,1603.03827v1,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",3,Recent approaches based on artificial neural n...,"[{'term': 'cs.CL', 'scheme': 'http://arxiv.org...",Sequential Short-Text Classification with Recu...,2016
2,"[{'name': 'Iulian Vlad Serban'}, {'name': 'Tim...",2,1606.00776v2,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",6,We introduce the multiresolution recurrent neu...,"[{'term': 'cs.CL', 'scheme': 'http://arxiv.org...",Multiresolution Recurrent Neural Networks: An ...,2016
3,"[{'name': 'Sebastian Ruder'}, {'name': 'Joachi...",23,1705.08142v2,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",5,Multi-task learning is motivated by the observ...,"[{'term': 'stat.ML', 'scheme': 'http://arxiv.o...",Learning what to share between loosely related...,2017
4,"[{'name': 'Iulian V. Serban'}, {'name': 'Chinn...",7,1709.02349v2,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",9,We present MILABOT: a deep reinforcement learn...,"[{'term': 'cs.CL', 'scheme': 'http://arxiv.org...",A Deep Reinforcement Learning Chatbot,2017


In [5]:
df2 = pd.DataFrame(df.author.str.split('}').tolist(),index = df.index).stack()
df2.head()

0  0            [{'name': 'Ahmed Osman'
   1        , {'name': 'Wojciech Samek'
   2                                  ]
1  0           [{'name': 'Ji Young Lee'
   1    , {'name': 'Franck Dernoncourt'
dtype: object

In [6]:
def rem_unwanted(line):
    return re.sub("\'term'|\'rel'|\'href'|\'type'|\'title'|\[|\{|\'name'|\'|\]|\,|\}",'',line).strip(' ').strip("''").strip(":")

In [7]:
df2 = pd.DataFrame(df2.apply(rem_unwanted))

In [8]:
df2 = pd.DataFrame(df2.unstack().iloc[:,0:2].to_records()).drop(columns={'index'})

In [9]:
df2.columns = ['Author1','Author2']

In [10]:
df2.Author1 = df2.Author1.str.strip(' ')
df2.Author2 = df2.Author2.str.strip(' ')

In [11]:
df2[df2.Author2 == '']

,Author1,Author2
9,Tony Beltramelli,
20,Zachary C. Lipton,
55,Jiwei Li,
75,Dirk Weissenborn,
118,Matt Shannon,
...,...,...
40909,Mithun Das Gupta,
40912,Vladimir Pestov,
40921,Yohji Akama,
40923,Giovanni Zappella,


In [12]:
df2 = df2.reset_index().drop(columns='index')

In [13]:
df2.head()

,Author1,Author2
0,Ahmed Osman,Wojciech Samek
1,Ji Young Lee,Franck Dernoncourt
2,Iulian Vlad Serban,Tim Klinger
3,Sebastian Ruder,Joachim Bingel
4,Iulian V. Serban,Chinnadhurai Sankar


In [14]:
len(df2.Author1.unique())

24707

In [15]:
df3 = pd.DataFrame(df.link.str.split(', ').tolist(),index = df.index).stack()

In [16]:
df3 = pd.DataFrame(df3.apply(rem_unwanted,convert_dtype=True))

In [17]:
df3.head()

0
0 0                           alternate
  1   http://arxiv.org/abs/1802.00209v1
  2                           text/html
  3                             related
  4   http://arxiv.org/pdf/1802.00209v1

In [18]:
df3 = df3.unstack()

In [19]:
df3.head()

0                                                            \
           0                                   1           2         3    
0   alternate   http://arxiv.org/abs/1802.00209v1   text/html   related   
1   alternate   http://arxiv.org/abs/1603.03827v1   text/html   related   
2   alternate   http://arxiv.org/abs/1606.00776v2   text/html   related   
3   alternate   http://arxiv.org/abs/1705.08142v2   text/html   related   
4   alternate   http://arxiv.org/abs/1709.02349v2   text/html   related   

                                                                              \
                                   4                 5     6    7    8    9    
0   http://arxiv.org/pdf/1802.00209v1   application/pdf   pdf  NaN  NaN  NaN   
1   http://arxiv.org/pdf/1603.03827v1   application/pdf   pdf  NaN  NaN  NaN   
2   http://arxiv.org/pdf/1606.00776v2   application/pdf   pdf  NaN  NaN  NaN   
3   http://arxiv.org/pdf/1705.08142v2   application/pdf   pdf  NaN  NaN  NaN   
4   http://arxiv.org/pdf/1709.02349v2   application/pdf   pdf  NaN  NaN  NaN   

   ...                                                    
   ...   25   26   27   28   29   30   31   32   33   34  
0  ...  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
1  ...  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
2  ...  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
3  ...  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
4  ...  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  

[5 rows x 35 columns]

In [20]:
links = df3.iloc[:,[1,4]]
links.columns = ['textLink','pdfLink']
links

,textLink,pdfLink
0,http://arxiv.org/abs/1802.00209v1,http://arxiv.org/pdf/1802.00209v1
1,http://arxiv.org/abs/1603.03827v1,http://arxiv.org/pdf/1603.03827v1
2,http://arxiv.org/abs/1606.00776v2,http://arxiv.org/pdf/1606.00776v2
3,http://arxiv.org/abs/1705.08142v2,http://arxiv.org/pdf/1705.08142v2
4,http://arxiv.org/abs/1709.02349v2,http://arxiv.org/pdf/1709.02349v2
...,...,...
40995,http://arxiv.org/abs/1404.4702v2,http://arxiv.org/pdf/1404.4702v2
40996,http://arxiv.org/abs/1404.5421v1,http://arxiv.org/pdf/1404.5421v1
40997,http://arxiv.org/abs/1404.5899v1,http://arxiv.org/pdf/1404.5899v1
40998,http://dx.doi.org/10.1007/978-3-319-08434-3_8,alternate


In [21]:
tags = pd.DataFrame(df['tag'].str.split(',').tolist())
# tags = tags[tags.str.contains('term')]
# tags.head()
tags = tags.iloc[:,[0,3,6]].stack()
# tags = tags.iloc[:,0]

In [22]:
tags.head()

0  0    [{'term': 'cs.AI'
   3     {'term': 'cs.CL'
   6     {'term': 'cs.CV'
1  0    [{'term': 'cs.CL'
   3     {'term': 'cs.AI'
dtype: object

In [23]:
tags = tags.apply(rem_unwanted)

In [24]:
tags = tags.unstack()

In [25]:
tags[0] = tags[0].str.strip()
tags.iloc[:,1] = tags.iloc[:,1].str.strip()
tags.iloc[:,2] = tags.iloc[:,2].str.strip()

In [26]:
tags.columns = ['Topic1','Topic2','Topic3']

In [27]:
tags.head()

,Topic1,Topic2,Topic3
0,cs.AI,cs.CL,cs.CV
1,cs.CL,cs.AI,cs.LG
2,cs.CL,cs.AI,cs.LG
3,stat.ML,cs.AI,cs.CL
4,cs.CL,cs.AI,cs.LG


In [28]:
pre0 = pd.merge(df,tags,how = 'inner',left_index=True,right_index=True).drop('tag',axis=1)
pre = pd.merge(pre0,df2,how = 'inner',left_index=True,right_index=True).drop('author',axis=1)
data = pd.merge(pre,links,how = 'inner',left_index=True,right_index=True).drop('link',axis=1)

In [29]:
def rem_bracket(line):
    return line.strip(')')

In [30]:
df.head()

,author,day,id,link,month,summary,tag,title,year
0,"[{'name': 'Ahmed Osman'}, {'name': 'Wojciech S...",1,1802.00209v1,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",2,We propose an architecture for VQA which utili...,"[{'term': 'cs.AI', 'scheme': 'http://arxiv.org...",Dual Recurrent Attention Units for Visual Ques...,2018
1,"[{'name': 'Ji Young Lee'}, {'name': 'Franck De...",12,1603.03827v1,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",3,Recent approaches based on artificial neural n...,"[{'term': 'cs.CL', 'scheme': 'http://arxiv.org...",Sequential Short-Text Classification with Recu...,2016
2,"[{'name': 'Iulian Vlad Serban'}, {'name': 'Tim...",2,1606.00776v2,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",6,We introduce the multiresolution recurrent neu...,"[{'term': 'cs.CL', 'scheme': 'http://arxiv.org...",Multiresolution Recurrent Neural Networks: An ...,2016
3,"[{'name': 'Sebastian Ruder'}, {'name': 'Joachi...",23,1705.08142v2,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",5,Multi-task learning is motivated by the observ...,"[{'term': 'stat.ML', 'scheme': 'http://arxiv.o...",Learning what to share between loosely related...,2017
4,"[{'name': 'Iulian V. Serban'}, {'name': 'Chinn...",7,1709.02349v2,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",9,We present MILABOT: a deep reinforcement learn...,"[{'term': 'cs.CL', 'scheme': 'http://arxiv.org...",A Deep Reinforcement Learning Chatbot,2017


In [31]:
data.head()

,day,id,month,summary,title,year,Topic1,Topic2,Topic3,Author1,Author2,textLink,pdfLink
0,1,1802.00209v1,2,We propose an architecture for VQA which utili...,Dual Recurrent Attention Units for Visual Ques...,2018,cs.AI,cs.CL,cs.CV,Ahmed Osman,Wojciech Samek,http://arxiv.org/abs/1802.00209v1,http://arxiv.org/pdf/1802.00209v1
1,12,1603.03827v1,3,Recent approaches based on artificial neural n...,Sequential Short-Text Classification with Recu...,2016,cs.CL,cs.AI,cs.LG,Ji Young Lee,Franck Dernoncourt,http://arxiv.org/abs/1603.03827v1,http://arxiv.org/pdf/1603.03827v1
2,2,1606.00776v2,6,We introduce the multiresolution recurrent neu...,Multiresolution Recurrent Neural Networks: An ...,2016,cs.CL,cs.AI,cs.LG,Iulian Vlad Serban,Tim Klinger,http://arxiv.org/abs/1606.00776v2,http://arxiv.org/pdf/1606.00776v2
3,23,1705.08142v2,5,Multi-task learning is motivated by the observ...,Learning what to share between loosely related...,2017,stat.ML,cs.AI,cs.CL,Sebastian Ruder,Joachim Bingel,http://arxiv.org/abs/1705.08142v2,http://arxiv.org/pdf/1705.08142v2
4,7,1709.02349v2,9,We present MILABOT: a deep reinforcement learn...,A Deep Reinforcement Learning Chatbot,2017,cs.CL,cs.AI,cs.LG,Iulian V. Serban,Chinnadhurai Sankar,http://arxiv.org/abs/1709.02349v2,http://arxiv.org/pdf/1709.02349v2


In [32]:
tags = pd.read_csv('../input/tagsdesc/tagsdesc.txt',sep='/n',header=None,engine='python')

In [33]:
tags.head()

,0
0,cs.AI - Artificial Intelligence
1,"Covers all areas of AI except Vision, Robotics..."
2,cs.CL - Computation and Language
3,Covers natural language processing. Roughly in...
4,cs.CC - Computational Complexity


In [34]:
tags.tail()

,0
204,"Dynamical systems, chaos, quantum chaos, topol..."
205,nlin.SI - Exactly Solvable and Integrable Systems
206,"Exactly solvable systems, integrable PDEs, int..."
207,nlin.PS - Pattern Formation and Solitons
208,"Pattern formation, coherent structures, solitons"


In [35]:
d1 = pd.DataFrame(tags.iloc[[i for i in tags.index if i%2==0]].reset_index().iloc[0:47][0].str.split(' - ').tolist())

In [36]:
d1.head()

,0,1
0,cs.AI,Artificial Intelligence
1,cs.CL,Computation and Language
2,cs.CC,Computational Complexity
3,cs.CE,"Computational Engineering, Finance, and Science"
4,cs.CG,Computational Geometry


In [37]:
d1['TopicExplain'] = tags.iloc[[i for i in tags.index if i%2!=0]][0].reset_index()[0]

In [38]:
d1.columns = ['Topic','FullTopic','TopicExplain']

In [39]:
tags = d1.copy()

In [40]:
tags.head()

,Topic,FullTopic,TopicExplain
0,cs.AI,Artificial Intelligence,"Covers all areas of AI except Vision, Robotics..."
1,cs.CL,Computation and Language,Covers natural language processing. Roughly in...
2,cs.CC,Computational Complexity,"Covers models of computation, complexity class..."
3,cs.CE,"Computational Engineering, Finance, and Science",Covers applications of computer science to the...
4,cs.CG,Computational Geometry,Roughly includes material in ACM Subject Class...


In [41]:
data.fillna(' ',inplace=True)

In [42]:
data.isna().any()

day         False
id          False
month       False
summary     False
title       False
year        False
Topic1      False
Topic2      False
Topic3      False
Author1     False
Author2     False
textLink    False
pdfLink     False
dtype: bool

In [43]:
data['summary'][0]

'We propose an architecture for VQA which utilizes recurrent layers to\ngenerate visual and textual attention. The memory characteristic of the\nproposed recurrent attention units offers a rich joint embedding of visual and\ntextual features and enables the model to reason relations between several\nparts of the image and question. Our single model outperforms the first place\nwinner on the VQA 1.0 dataset, performs within margin to the current\nstate-of-the-art ensemble model. We also experiment with replacing attention\nmechanisms in other state-of-the-art models with our implementation and show\nincreased accuracy. In both cases, our recurrent attention mechanism improves\nperformance in tasks requiring sequential or relational reasoning on the VQA\ndataset.'

In [44]:
data['title'][0]

'Dual Recurrent Attention Units for Visual Question Answering'

In [45]:
def rem_n(line):
    return re.sub('\\n',' ',line)

In [46]:
data['summary'] = data['summary'].apply(rem_n)
data['title'] = data['title'].apply(rem_n)

In [47]:
data['summary'][0]

'We propose an architecture for VQA which utilizes recurrent layers to generate visual and textual attention. The memory characteristic of the proposed recurrent attention units offers a rich joint embedding of visual and textual features and enables the model to reason relations between several parts of the image and question. Our single model outperforms the first place winner on the VQA 1.0 dataset, performs within margin to the current state-of-the-art ensemble model. We also experiment with replacing attention mechanisms in other state-of-the-art models with our implementation and show increased accuracy. In both cases, our recurrent attention mechanism improves performance in tasks requiring sequential or relational reasoning on the VQA dataset.'

In [48]:
db1 = pd.merge(data,tags,how='left',left_on='Topic1',right_on='Topic').drop(columns=['Topic'])
db2 = pd.merge(db1,tags,how='left',left_on='Topic2',right_on='Topic').drop(columns=['Topic','TopicExplain_y'])
db3 = pd.merge(db2,tags,how='left',left_on='Topic3',right_on='Topic').drop(columns=['Topic','TopicExplain'])

In [49]:
db3 = db3[['id', 'summary', 'title', 'year', 'FullTopic_x', 'FullTopic_y', 'FullTopic','TopicExplain_x', 'Topic1', 'Topic2', 'Topic3','Author1', 'Author2', 'textLink', 'pdfLink']]

In [50]:
db3.head()

,id,summary,title,year,FullTopic_x,FullTopic_y,FullTopic,TopicExplain_x,Topic1,Topic2,Topic3,Author1,Author2,textLink,pdfLink
0,1802.00209v1,We propose an architecture for VQA which utili...,Dual Recurrent Attention Units for Visual Ques...,2018,Artificial Intelligence,Computation and Language,Computer Vision and Pattern Recognition,"Covers all areas of AI except Vision, Robotics...",cs.AI,cs.CL,cs.CV,Ahmed Osman,Wojciech Samek,http://arxiv.org/abs/1802.00209v1,http://arxiv.org/pdf/1802.00209v1
1,1603.03827v1,Recent approaches based on artificial neural n...,Sequential Short-Text Classification with Recu...,2016,Computation and Language,Artificial Intelligence,Machine Learning,Covers natural language processing. Roughly in...,cs.CL,cs.AI,cs.LG,Ji Young Lee,Franck Dernoncourt,http://arxiv.org/abs/1603.03827v1,http://arxiv.org/pdf/1603.03827v1
2,1606.00776v2,We introduce the multiresolution recurrent neu...,Multiresolution Recurrent Neural Networks: An ...,2016,Computation and Language,Artificial Intelligence,Machine Learning,Covers natural language processing. Roughly in...,cs.CL,cs.AI,cs.LG,Iulian Vlad Serban,Tim Klinger,http://arxiv.org/abs/1606.00776v2,http://arxiv.org/pdf/1606.00776v2
3,1705.08142v2,Multi-task learning is motivated by the observ...,Learning what to share between loosely related...,2017,NaN,Artificial Intelligence,Computation and Language,NaN,stat.ML,cs.AI,cs.CL,Sebastian Ruder,Joachim Bingel,http://arxiv.org/abs/1705.08142v2,http://arxiv.org/pdf/1705.08142v2
4,1709.02349v2,We present MILABOT: a deep reinforcement learn...,A Deep Reinforcement Learning Chatbot,2017,Computation and Language,Artificial Intelligence,Machine Learning,Covers natural language processing. Roughly in...,cs.CL,cs.AI,cs.LG,Iulian V. Serban,Chinnadhurai Sankar,http://arxiv.org/abs/1709.02349v2,http://arxiv.org/pdf/1709.02349v2


In [51]:
db3.columns = ['id', 'summary', 'title', 'year','Topic1',
       'Topic2', 'Topic3', 'Topic', 'DTopic1', 'DTopic2', 'DTopic3',
       'Author1', 'Author2', 'textLink', 'pdfLink' ]

In [52]:
db3.drop(columns=['DTopic1','DTopic2','DTopic3'],inplace=True)

In [53]:
f1 = db3.copy()

In [54]:
f1.fillna(' ',inplace=True)

In [55]:
f1.isna().sum()

id          0
summary     0
title       0
year        0
Topic1      0
Topic2      0
Topic3      0
Topic       0
Author1     0
Author2     0
textLink    0
pdfLink     0
dtype: int64

In [56]:
nlp = spacy.load('en',disable = ['ner','parser'])
spacy.require_gpu()

True

In [57]:
stopwords = list(STOP_WORDS)+list((''.join(string.punctuation)).strip(''))+['-pron-','-PRON-']
len(stopwords)

360

In [58]:
def lemmatizer(df):
    texts = []
    c=0
    for text in df:
        if c%1000==0:
            print("Processed articles: ",c)
        c+=1
        doc = nlp(text)
        lemma = [word.lemma_.lower().strip('') for word in doc]
        words = [word for word in lemma if word not in stopwords]
        texts.append(' '.join(words))
    return pd.Series(texts)

In [59]:
f1['Full'] = (f1['title']+" "+f1['summary']+' '+f1['Topic1']+' '+f1['Topic2']+' '+f1['Topic3']+' '+f1['Topic']+' '+f1['Author1']+' '+f1['Author2'])

In [60]:
t = time()
processed_text = lemmatizer(f1['Full'])

Processed articles:  0
Processed articles:  1000
Processed articles:  2000
Processed articles:  3000
Processed articles:  4000
Processed articles:  5000
Processed articles:  6000
Processed articles:  7000
Processed articles:  8000
Processed articles:  9000
Processed articles:  10000
Processed articles:  11000
Processed articles:  12000
Processed articles:  13000
Processed articles:  14000
Processed articles:  15000
Processed articles:  16000
Processed articles:  17000
Processed articles:  18000
Processed articles:  19000
Processed articles:  20000
Processed articles:  21000
Processed articles:  22000
Processed articles:  23000
Processed articles:  24000
Processed articles:  25000
Processed articles:  26000
Processed articles:  27000
Processed articles:  28000
Processed articles:  29000
Processed articles:  30000
Processed articles:  31000
Processed articles:  32000
Processed articles:  33000
Processed articles:  34000
Processed articles:  35000
Processed articles:  36000
Processed arti

In [61]:
(time()-t)/60

6.778923817475637

In [62]:
processed_text

0        dual recurrent attention unit visual question ...
1        sequential short text classification recurrent...
2        multiresolution recurrent neural networks appl...
3        learn share loosely related task multi task le...
4        deep reinforcement learn chatbot present milab...
                               ...                        
40995    nearly tight bounds \ell_1 approximation self ...
40996    concurrent bandit cognitive radio network cons...
40997    comparison cluster missing data methods health...
40998    apply machine learn problem choose heuristic  ...
40999    multi level data fusion approach speaker ident...
Length: 41000, dtype: object

In [63]:
processed_text.index = f1['title'].values

In [64]:
processed_text = pd.DataFrame(processed_text)

In [65]:
processed_text.iloc[0:12].index

Index(['Dual Recurrent Attention Units for Visual Question Answering',
       'Sequential Short-Text Classification with Recurrent and Convolutional   Neural Networks',
       'Multiresolution Recurrent Neural Networks: An Application to Dialogue   Response Generation',
       'Learning what to share between loosely related tasks',
       'A Deep Reinforcement Learning Chatbot',
       'Generating Sentences by Editing Prototypes',
       'A Deep Reinforcement Learning Chatbot (Short Version)',
       'Document Image Coding and Clustering for Script Discrimination',
       'Tutorial on Answering Questions about Images with Deep Learning',
       'pix2code: Generating Code from a Graphical User Interface Screenshot',
       'A Unified Deep Neural Network for Speaker and Language Recognition',
       'Efficient Neural Architecture Search via Parameter Sharing'],
      dtype='object')

In [66]:
processed_text.iloc[:6].values

array([['dual recurrent attention unit visual question answering propose architecture vqa utilize recurrent layer generate visual textual attention memory characteristic propose recurrent attention unit offer rich joint embedding visual textual feature enable model reason relation image question single model outperform place winner vqa 1.0 dataset perform margin current state art ensemble model experiment replace attention mechanism state art model implementation increase accuracy case recurrent attention mechanism improve performance task require sequential relational reasoning vqa dataset artificial intelligence computation language computer vision pattern recognition cover area ai vision robotics machine learning multiagent systems computation language natural language processing separate subject area particular include expert systems theorem proving overlap logic computer science knowledge representation planning uncertainty ai roughly include material acm subject classes i.2.0 i.2

In [67]:
phr = [i[0].split() for i in processed_text.values]

In [68]:
phrases = Phrases(phr,min_count=10,progress_per=1000)

In [69]:
bigram = Phraser(phrases)

In [70]:
sentences = bigram[phr]

In [71]:
phrases = Phrases(sentences,min_count=5,progress_per=1000)

In [72]:
trigram = Phraser(phrases)

In [73]:
trigrams = trigram[sentences]

In [74]:
trigrams[0]

['dual',
 'recurrent_attention',
 'unit',
 'visual_question_answering',
 'propose',
 'architecture',
 'vqa',
 'utilize',
 'recurrent_layer',
 'generate',
 'visual_textual',
 'attention',
 'memory',
 'characteristic',
 'propose',
 'recurrent_attention',
 'unit',
 'offer',
 'rich',
 'joint_embedding',
 'visual_textual',
 'feature',
 'enable',
 'model',
 'reason',
 'relation',
 'image',
 'question',
 'single',
 'model',
 'outperform',
 'place',
 'winner',
 'vqa',
 '1.0',
 'dataset',
 'perform',
 'margin',
 'current_state_art',
 'ensemble',
 'model',
 'experiment',
 'replace',
 'attention_mechanism',
 'state_art',
 'model',
 'implementation',
 'increase',
 'accuracy',
 'case',
 'recurrent',
 'attention_mechanism',
 'improve_performance',
 'task',
 'require',
 'sequential',
 'relational_reasoning',
 'vqa_dataset',
 'artificial_intelligence',
 'computation_language',
 'computer_vision_pattern_recognition',
 'cover_area_ai_vision',
 'robotics_machine_learning_multiagent',
 'systems_computatio

In [75]:
tagged_data = [TaggedDocument(words=' '.join(i),tags=[j]) for i, j in zip(trigrams,processed_text.index)]

In [76]:
tagged_data[0]

TaggedDocument(words='dual recurrent_attention unit visual_question_answering propose architecture vqa utilize recurrent_layer generate visual_textual attention memory characteristic propose recurrent_attention unit offer rich joint_embedding visual_textual feature enable model reason relation image question single model outperform place winner vqa 1.0 dataset perform margin current_state_art ensemble model experiment replace attention_mechanism state_art model implementation increase accuracy case recurrent attention_mechanism improve_performance task require sequential relational_reasoning vqa_dataset artificial_intelligence computation_language computer_vision_pattern_recognition cover_area_ai_vision robotics_machine_learning_multiagent systems_computation_language natural_language_processing_separate subject_area_particular_include expert_systems_theorem_proving overlap_logic_computer_science knowledge_representation_planning_uncertainty ai_roughly_include_material acm_subject_clas

In [77]:
docmodel = Doc2Vec(dm=1,window=2, workers=4, negative=5,vector_size = 300,min_count=3,dbow_words=1)

In [78]:
docmodel.build_vocab(tagged_data)

In [79]:
docmodel.corpus_total_words

42238798

In [80]:
t = time()
docmodel.train(tagged_data,total_examples=docmodel.corpus_count,epochs=100)

In [81]:
(time()-t)/60

37.69581012328466

In [82]:
docmodel.init_sims(replace=True)
docmodel.save('model')

In [83]:
docmodel = Doc2Vec.load('model')

In [108]:

tokens =list(input("Enter your keywords: "))

new_vector = docmodel.infer_vector(tokens)
top=docmodel.docvecs.most_similar([new_vector],topn=10)
top

Enter your keywords:  artificial intelligence


[('Theoretical Issues for Global Cumulative Treatment Analysis (GCTA)',
  0.4885091185569763),
 ('A Model of Free Will for Artificial Entities', 0.4638223648071289),
 ('Iterated Support Vector Machines for Distance Metric Learning',
  0.4478147029876709),
 ('End-to-end representation learning for Correlation Filter based tracking',
  0.4402959942817688),
 ('Machine Learning for Dental Image Analysis', 0.4391498863697052),
 ('Variational Bayesian inference for linear and logistic regression',
  0.43816137313842773),
 ('Emergence of foveal image sampling from learning to attend in visual   scenes',
  0.4352482557296753),
 ('Learning Local Metrics and Influential Regions for Classification',
  0.43308496475219727),
 ('On a Possible Similarity between Gene and Semantic Networks',
  0.43208324909210205),
 ('Computational approach to the emergence and evolution of language -   evolutionary naming game model',
  0.43183547258377075)]

In [114]:
j = top
r = f1[f1['Full'].isin(top)]
p = ['Searched',]*len(r)
for i in j:
     r = pd.concat([r,f1[f1['title']==i[0]]])
     p.append(i[1])
r['Similarity Score'] = p
s=r.copy()

HTML(r.to_html(escape=False,render_links=True))

,id,summary,title,year,Topic1,Topic2,Topic3,Topic,Author1,Author2,textLink,pdfLink,Full,Similarity Score
40564,1308.1066v1,"Adaptive trials are now mainstream science. Recently, researchers have taken the adaptive trial concept to its natural conclusion, proposing what we call ""Global Cumulative Treatment Analysis"" (GCTA). Similar to the adaptive trial, decision making and data collection and analysis in the GCTA are continuous and integrated, and treatments are ranked in accord with the statistics of this information, combined with what offers the most information gain. Where GCTA differs from an adaptive trial, or, for that matter, from any trial design, is that all patients are implicitly participants in the GCTA process, regardless of whether they are formally enrolled in a trial. This paper discusses some of the theoretical and practical issues that arise in the design of a GCTA, along with some preliminary thoughts on how they might be approached.",Theoretical Issues for Global Cumulative Treatment Analysis (GCTA),2013,,Machine Learning,,,Jeff Shrager,,http://arxiv.org/abs/1308.1066v1,http://arxiv.org/pdf/1308.1066v1,"Theoretical Issues for Global Cumulative Treatment Analysis (GCTA) Adaptive trials are now mainstream science. Recently, researchers have taken the adaptive trial concept to its natural conclusion, proposing what we call ""Global Cumulative Treatment Analysis"" (GCTA). Similar to the adaptive trial, decision making and data collection and analysis in the GCTA are continuous and integrated, and treatments are ranked in accord with the statistics of this information, combined with what offers the most information gain. Where GCTA differs from an adaptive trial, or, for that matter, from any trial design, is that all patients are implicitly participants in the GCTA process, regardless of whether they are formally enrolled in a trial. This paper discusses some of the theoretical and practical issues that arise in the design of a GCTA, along with some preliminary thoughts on how they might be approached. Machine Learning Jeff Shrager",0.488509
37751,1802.09317v1,"The impression of free will is the feeling according to which our choices are neither imposed from our inside nor from outside. It is the sense we are the ultimate cause of our acts. In direct opposition with the universal determinism, the existence of free will continues to be discussed. In this paper, free will is linked to a decisional mechanism: an agent is provided with free will if having performed a predictable choice Cp, it can immediately perform another choice Cr in a random way. The intangible feeling of free will is replaced by a decision-making process including a predictable decision-making process immediately followed by an unpredictable decisional one.",A Model of Free Will for Artificial Entities,2018,Multiagent Systems,Artificial Intelligence,,"Covers multiagent systems, distributed artificial intelligence, intelligent agents, coordinated interactions. and practical applications. Roughly covers ACM Subject Class I.2.11.",Eric Sanchis,,http://arxiv.org/abs/1802.09317v1,http://arxiv.org/pdf/1802.09317v1,"A Model of Free Will for Artificial Entities The impression of free will is the feeling according to which our choices are neither imposed from our inside nor from outside. It is the sense we are the ultimate cause of our acts. In direct opposition with the universal determinism, the existence of free will continues to be discussed. In this paper, free will is linked to a decisional mechanism: an agent is provided with free will if having performed a predictable choice Cp, it can immediately perform another choice Cr in a random way. The intangible feeling of free will is replaced by a decision-making process including a predictable decision-making process immediately followed by an unpredictable decisional one. Multiagent Systems Artificial Intelligence Covers multiagent systems, distributed artificial intelligence, intelligent agents